In [1]:
from Bio import Entrez
#add email to get 
Entrez.email = "nkovalchik98@gmail.com"  

def fetch_fasta(accession, out_fname):
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
    seq = handle.read()
    handle.close()
    with open(out_fname, "w") as f:
        f.write(seq)
    print("Wrote:", out_fname)

fetch_fasta("NC_012920.1", "rCRS.fasta")
fetch_fasta("KY645908.1", "sample_KY645908.fasta")



Wrote: rCRS.fasta
Wrote: sample_KY645908.fasta


In [2]:
import os
from Bio import SeqIO
from Bio.Seq import Seq
import pandas as pd

print("cwd:", os.getcwd())


cwd: C:\Users\nkovalchik


In [3]:
ref_file = "rCRS.fasta" #Normal DNA 
sample_file = "sample_KY645908.fasta" #Sample DNA from someone

ref_rec = SeqIO.read(ref_file, "fasta")
sample_rec = SeqIO.read(sample_file, "fasta")

ref_seq = str(ref_rec.seq).upper()
sample_seq = str(sample_rec.seq).upper()

print("Ref id:", ref_rec.id, "len:", len(ref_seq))
print("Sample id:", sample_rec.id, "len:", len(sample_seq))


Ref id: NC_012920.1 len: 16569
Sample id: KY645908.1 len: 16569


In [4]:
COMPARE_START = 16024
COMPARE_END   = 16383

print("Comparing positions", COMPARE_START, "to", COMPARE_END)
ref_region = ref_seq[COMPARE_START-1:COMPARE_END]
sample_region = sample_seq[COMPARE_START-1:COMPARE_END]

print("Ref id:", ref_rec.id, "len:", len(ref_seq))
print("Sample id:", sample_rec.id, "len:", len(sample_seq))


Comparing positions 16024 to 16383
Ref id: NC_012920.1 len: 16569
Sample id: KY645908.1 len: 16569


In [5]:
snps = []
for i, (r, s) in enumerate(zip(ref_region, sample_region), start=COMPARE_START):
    if r != s:
        snps.append({"pos": i, "ref": r, "alt": s})


if len(ref_region) != len(sample_region):
    longer, name = (ref_region, "ref") if len(ref_region) > len(sample_region) else (sample_region, "sample")
    diff_start = COMPARE_START + min(len(ref_region), len(sample_region))
    for idx in range(min(len(ref_region), len(sample_region)), len(longer)):
        pos = COMPARE_START + idx
        base_ref = ref_region[idx] if idx < len(ref_region) else "-"
        base_sample = sample_region[idx] if idx < len(sample_region) else "-"
        snps.append({"pos": pos, "ref": base_ref, "alt": base_sample})

print(f"Found {len(snps)} differences.")
df = pd.DataFrame(snps)
df.head(30)


Found 4 differences.


,pos,ref,alt
0,16142,C,T
1,16223,C,T
2,16325,T,C
3,16362,T,C


In [6]:
def is_transition(a,b):
    pairs = {("A","G"),("G","A"),("C","T"),("T","C")}
    return (a,b) in pairs

transitions = sum(is_transition(row["ref"], row["alt"]) for _,row in df.iterrows())
transversions = len(df) - transitions

print("Total differences:", len(df))
print("Transitions:", transitions) #A mutation that switches within the same chemical class 1 ring to 1 rings or 2 ring to 2 ring. Milder mutation
print("Transversions:", transversions) #A mutation that switches between classes 1 ring to 2 ring or 2 ring to 1 ring. More savier.


Total differences: 4
Transitions: 4
Transversions: 0


In [7]:
# Cell 6 - adjust to the gene coordinates you want to analyze
GENE_START = 3307
GENE_END   = 4262

# extract CDS slices (must be same length and in-frame for simple check)
cds_ref = ref_seq[GENE_START-1:GENE_END]
cds_sample = sample_seq[GENE_START-1:GENE_END]

# if lengths mismatch, trim to same length that is multiple of 3
L = min(len(cds_ref), len(cds_sample)) - (min(len(cds_ref), len(cds_sample)) % 3)
cds_ref = cds_ref[:L]
cds_sample = cds_sample[:L]

prot_ref = Seq(cds_ref).translate(table=2)      # mitochondrial table
prot_sample = Seq(cds_sample).translate(table=2)

# find AA differences
aa_changes = []
for i, (ra, sa) in enumerate(zip(prot_ref, prot_sample), start=1):
    if ra != sa:
        aa_changes.append({"aa_pos": i, "ref_aa": ra, "alt_aa": sa})

print("Protein length (AA):", len(prot_ref))
print("AA changes:", len(aa_changes))
pd.DataFrame(aa_changes).head(20)


Protein length (AA): 318
AA changes: 0


""
